# VideoRecommendationEngine

In [8]:
import numpy as np
import scipy.sparse as sp
import faiss
import time

class VideoRecommendationEngine:
    def __init__(self, matrix_path):
        self.matrix_path = matrix_path
        
        # 1. Load ma trận thưa hiện tại từ file (Ví dụ: TF-IDF matrix)
        print(f"[Init] Đang load dữ liệu từ {self.matrix_path}...")
        self.matrix = sp.load_npz(self.matrix_path)
        
        # Ép kiểu về CSR format để truy xuất theo hàng (row) nhanh hơn
        self.matrix = self.matrix.tocsr() 
        num_videos, num_features = self.matrix.shape
        print(f"[Init] Kích thước ma trận gốc: {num_videos} videos, {num_features} features")

        # ==========================================
        # TÍCH HỢP FAISS
        # ==========================================
        print("[Init] Đang xây dựng FAISS Index...")
        start_time = time.time()
        
        # 2. Chuyển Sparse -> Dense và ép kiểu float32 (BẮT BUỘC cho FAISS)
        # Lưu ý: Nếu ma trận quá khổng lồ, thường người ta sẽ dùng TruncatedSVD
        # để giảm số chiều (features) xuống khoảng 300 trước khi đưa vào FAISS.
        dense_matrix = self.matrix.toarray().astype(np.float32)
        
        # 3. Chuẩn hóa L2 (L2 Normalize) trực tiếp bằng hàm tối ưu C++ của FAISS
        faiss.normalize_L2(dense_matrix)
        
        # 4. Khởi tạo Index tính Inner Product (Cosine Similarity)
        self.index = faiss.IndexFlatIP(num_features)
        
        # 5. Nạp dữ liệu vào RAM
        self.index.add(dense_matrix)
        
        print(f"[Init] Hoàn tất build FAISS trong {time.time() - start_time:.2f}s")
        print(f"[Init] Số lượng video trong hệ thống nhận diện (FAISS): {self.index.ntotal}")

    def get_similar_videos(self, video_id, top_k=5):
        """Tìm các video giống với một video CŨ đã có trong hệ thống"""
        # Lấy vector của video_id từ ma trận sparse, chuyển sang dense float32
        query_vector_sparse = self.matrix[video_id]
        query_vector_dense = query_vector_sparse.toarray().astype(np.float32)
        
        # Chuẩn hóa vector truy vấn
        faiss.normalize_L2(query_vector_dense)
        
        # FAISS search (Lấy top_k + 1 vì kết quả đầu tiên thường chính là video_id đó)
        similarities, indices = self.index.search(query_vector_dense, top_k + 1)
        
        # Loại bỏ chính nó ra khỏi kết quả
        final_scores = []
        final_indices = []
        for i in range(len(indices[0])):
            if indices[0][i] != video_id:
                final_indices.append(indices[0][i])
                final_scores.append(similarities[0][i])
                
        return final_indices[:top_k], final_scores[:top_k]

    def add_new_video(self, new_sparse_vector):
        """
        Cập nhật video MỚI vào hệ thống.
        new_sparse_vector: scipy sparse matrix có shape (1, num_features)
        """
        # 1. Xử lý cho FAISS (Để tìm kiếm real-time)
        new_dense_vector = new_sparse_vector.toarray().astype(np.float32)
        faiss.normalize_L2(new_dense_vector)
        self.index.add(new_dense_vector) # Add vào FAISS chớp nhoáng O(1)
        
        # 2. Xử lý đồng bộ (Sync) với kho dữ liệu gốc (self.matrix)
        # Để lần sau khi khởi động lại server, bạn sp.save_npz() nó không bị mất data
        self.matrix = sp.vstack([self.matrix, new_sparse_vector]).tocsr()
        
        new_video_id = self.matrix.shape[0] - 1
        print(f"[Update] Đã thêm thành công! ID của video mới là: {new_video_id}")
        print(f"[Update] Tổng số lượng index hiện tại: {self.index.ntotal}")
        
        return new_video_id

In [11]:
candidates = [
    Path("data/processed/tfidf_matrix.npz"),
    Path("../data/processed/tfidf_matrix.npz"),
]

file_path = None
for p in candidates:
    if p.exists():
        file_path = str(p)
        break

if file_path is None:
    cwd = Path.cwd()
    raise FileNotFoundError(
        f"Không tìm thấy file tfidf_matrix.npz. CWD hiện tại: {cwd}. "
        "Đã thử: data/processed/tfidf_matrix.npz và ../data/processed/tfidf_matrix.npz"
    )

print(f"[Main] Dùng file: {file_path}")
matrix_path = file_path
engine = VideoRecommendationEngine(matrix_path=matrix_path)


[Main] Dùng file: ..\data\processed\tfidf_matrix.npz
[Init] Đang load dữ liệu từ ..\data\processed\tfidf_matrix.npz...
[Init] Kích thước ma trận gốc: 46625 videos, 10000 features
[Init] Đang xây dựng FAISS Index...
[Init] Hoàn tất build FAISS trong 1.61s
[Init] Số lượng video trong hệ thống nhận diện (FAISS): 46625


In [13]:
engine.get_similar_videos(video_id=0, top_k=50)

([np.int64(25950),
  np.int64(15444),
  np.int64(22063),
  np.int64(3012),
  np.int64(1823),
  np.int64(2154),
  np.int64(19227),
  np.int64(25948),
  np.int64(1895),
  np.int64(36463),
  np.int64(36462),
  np.int64(24668),
  np.int64(8279),
  np.int64(24670),
  np.int64(25848),
  np.int64(37871),
  np.int64(1894),
  np.int64(2214),
  np.int64(42880),
  np.int64(1896),
  np.int64(1639),
  np.int64(2697),
  np.int64(7593),
  np.int64(22856),
  np.int64(3896),
  np.int64(24405),
  np.int64(29858),
  np.int64(12248),
  np.int64(38711),
  np.int64(34304),
  np.int64(59),
  np.int64(22550),
  np.int64(44392),
  np.int64(8560),
  np.int64(7612),
  np.int64(589),
  np.int64(32006),
  np.int64(16425),
  np.int64(16255),
  np.int64(19225),
  np.int64(38924),
  np.int64(4750),
  np.int64(37609),
  np.int64(15811),
  np.int64(46070),
  np.int64(13101),
  np.int64(21887),
  np.int64(8935),
  np.int64(16560),
  np.int64(29848)],
 [np.float32(0.6590689),
  np.float32(0.59747946),
  np.float32(0.5444